# Comparative Analysis of Transfer Learning Strategies on Caltech-101 Using Pre-Trained CNN Architectures

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split
from sklearn.metrics import accuracy_score
import numpy as np

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

In [ ]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

In [ ]:
dataset_path = "/content/drive/MyDrive/caltech-101/101_ObjectCategories" # Corrected path to mounted Google Drive folder

dataset = datasets.ImageFolder(root=dataset_path, transform=transform)

num_classes = 102

## Dataset splitting and Preprocessing

In [ ]:
from sklearn.model_selection import train_test_split

SEED = 42

# Get labels and indices
targets = np.array(dataset.targets)
indices = np.arange(len(targets))

# ---- First split: 70% Train, 30% Temp ----
train_idx, temp_idx = train_test_split(
    indices,
    test_size=0.3,
    stratify=targets,
    random_state=SEED
)

# ---- Second split: 10% Val, 20% Test ----
val_idx, test_idx = train_test_split(
    temp_idx,
    test_size=0.66,
    stratify=targets[temp_idx],
    random_state=SEED
)

In [ ]:
from torch.utils.data import Subset

train_dataset = Subset(dataset, train_idx)
val_dataset   = Subset(dataset, val_idx)
test_dataset  = Subset(dataset, test_idx)

print("Train size:", len(train_dataset))
print("Validation size:", len(val_dataset))
print("Test size:", len(test_dataset))

In [ ]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=32, num_workers = 2, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=32, num_workers = 2, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=32, num_workers = 2, shuffle=False)

In [ ]:
from collections import Counter

train_labels = targets[train_idx]
val_labels   = targets[val_idx]
test_labels  = targets[test_idx]

print("Train distribution:", Counter(train_labels))
print("Val distribution:", Counter(val_labels))
print("Test distribution:", Counter(test_labels))

## Model Training and Evaluating functions

In [ ]:
def train_model(model, train_loader, val_loader,
                epochs=4, lr=1e-4, save_path="best_model.pth"):

    model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr
    )

    train_losses = []
    val_losses = []
    val_accuracies = []

    best_val_acc = 0

    for epoch in range(epochs):
        # ---- Training ----
        model.train()
        running_loss = 0.0

        for batch_idx, (images, labels) in enumerate(train_loader):
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)

            # --- Debugging checks ---
            if torch.isnan(outputs).any() or torch.isinf(outputs).any():
                print(f"Epoch {epoch+1}, Batch {batch_idx}: NaN or Inf found in model outputs!")
                # Optionally, break or handle this more gracefully
                # For now, we'll let it proceed to see if loss calculation is the exact point
            if labels.min() < 0 or labels.max() >= num_classes:
                print(f"Epoch {epoch+1}, Batch {batch_idx}: Labels out of expected range! Min: {labels.min().item()}, Max: {labels.max().item()}")
            # --- End Debugging checks ---

            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        train_loss = running_loss / len(train_loader)

        # ---- Validation ----
        model.eval()
        val_loss = 0.0
        correct = 0
        total = 0

        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)

                outputs = model(images)
                loss = criterion(outputs, labels)

                val_loss += loss.item()

                _, preds = torch.max(outputs, 1)
                correct += (preds == labels).sum().item()
                total += labels.size(0)

        val_loss /= len(val_loader)
        val_acc = correct / total

        train_losses.append(train_loss)
        val_losses.append(val_loss)
        val_accuracies.append(val_acc)

        print(f"Epoch [{epoch+1}/{epochs}] "
              f"Train Loss: {train_loss:.4f} | "
              f"Val Loss: {val_loss:.4f} | "
              f"Val Acc: {val_acc:.4f}")

        # ---- Save Best Model ----
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), save_path)

    print("Best Validation Accuracy:", best_val_acc)

    return model, train_losses, val_losses, val_accuracies

In [ ]:
from sklearn.metrics import accuracy_score, f1_score

def evaluate_model(model, test_loader):
    model.to(device)
    model.eval()

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            _, preds = torch.max(outputs, 1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)
    macro_f1 = f1_score(all_labels, all_preds, average='macro')

    print("Test Accuracy:", round(acc, 4))
    print("Test Macro F1:", round(macro_f1, 4))

    return acc, macro_f1

## Resnet-18 Model

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models

model = models.resnet18(pretrained=True)

num_classes = len(dataset.classes)
model.fc = nn.Linear(model.fc.in_features, num_classes)

for param in model.parameters():
    param.requires_grad = False

for param in model.fc.parameters():
    param.requires_grad = True

model = model.to(device)

In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-3
)

In [ ]:
model, train_losses, val_losses, val_accuracies = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=8,
    lr=1e-3,  
    save_path="resnet18_frozen_best.pth"
)

In [ ]:
model.load_state_dict(torch.load("resnet18_frozen_best.pth"))

test_acc, test_f1 = evaluate_model(model, test_loader)

print("Test Results (ResNet-18 Frozen)")
print(f"Test Accuracy : {test_acc*100:.2f}%")
print(f"Test Macro F1 : {test_f1:.4f}")

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models

model = models.resnet18(pretrained=True)

num_classes = len(dataset.classes)
model.fc = nn.Linear(model.fc.in_features, num_classes)

for param in model.parameters():
    param.requires_grad = False

for param in model.fc.parameters():
    param.requires_grad = True

for param in model.layer4.parameters():
    param.requires_grad = True

model = model.to(device)

In [ ]:
optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-3
)

In [ ]:
model, train_losses, val_losses, val_accuracies = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=8,
    lr=1e-4,
    save_path="resnet18_finetuned_best.pth"
)

In [ ]:
model.load_state_dict(torch.load("resnet18_finetuned_best.pth"))

test_acc, test_f1 = evaluate_model(model, test_loader)

print("Test Results (ResNet-18 Finetuned)")
print(f"Test Accuracy : {test_acc*100:.2f}%")
print(f"Test Macro F1 : {test_f1:.4f}")

## DenseNet121 Model

In [ ]:
from torchvision import models
import torch.nn as nn

model = models.densenet121(pretrained=True)

model.classifier = nn.Linear(model.classifier.in_features, num_classes)

for param in model.parameters():
    param.requires_grad = False

for param in model.classifier.parameters():
    param.requires_grad = True

model = model.to(device)

In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [ ]:
model, train_losses, val_losses, val_accuracies = train_model(model, train_loader, val_loader, epochs=8, lr=1e-4, save_path="densenet101_frozen_best.pth")

In [ ]:
model.load_state_dict(torch.load("densenet101_frozen_best.pth"))

test_acc, test_f1 = evaluate_model(model, test_loader)

print("Test Results (Densenet-101 Frozen)")
print(f"Test Accuracy : {test_acc*100:.2f}%")
print(f"Test Macro F1 : {test_f1:.4f}")

In [ ]:
from torchvision import models
import torch.nn as nn

model = models.densenet121(pretrained=True)

model.classifier = nn.Linear(model.classifier.in_features, num_classes)

for param in model.parameters():
    param.requires_grad = False

for param in model.classifier.parameters():
    param.requires_grad = True

for param in model.features.denseblock4.parameters():
    param.requires_grad = True

for param in model.features.norm5.parameters():
    param.requires_grad = True


model = model.to(device)

In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

In [ ]:
model, train_losses, val_losses, val_accuracies = train_model(model, train_loader, val_loader, epochs=8, lr=1e-4, save_path="densenet101_finetuned_best.pth")

In [ ]:
model.load_state_dict(torch.load("densenet101_finetuned_best.pth"))

test_acc, test_f1 = evaluate_model(model, test_loader)

print("Test Results (Densenet-101 Finetuned)")
print(f"Test Accuracy : {test_acc*100:.2f}%")
print(f"Test Macro F1 : {test_f1:.4f}")

## MobileNet V2 Model

In [ ]:
from torchvision import models
import torch.nn as nn

model = models.mobilenet_v2(pretrained=True)

model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)

for param in model.parameters():
    param.requires_grad = False

for param in model.classifier[1].parameters():
    param.requires_grad = True

model = model.to(device)

In [ ]:
optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-3
)

In [ ]:
model, train_losses, val_losses, val_accuracies = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=8,
    lr=1e-3,
    save_path="mobilenetv2_frozen_best.pth"
)

In [ ]:
model.load_state_dict(torch.load("mobilenetv2_frozen_best.pth"))

test_acc, test_f1 = evaluate_model(model, test_loader)

print("MobileNetV2 Frozen Results")
print(f"Test Accuracy : {test_acc*100:.2f}%")
print(f"Test Macro F1 : {test_f1:.4f}")

In [ ]:
from torchvision import models
import torch.nn as nn

model = models.mobilenet_v2(pretrained=True)

model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)

for param in model.parameters():
    param.requires_grad = False

for param in model.classifier[1].parameters():
    param.requires_grad = True

for param in model.features[-2:].parameters():
    param.requires_grad = True

model = model.to(device)

In [ ]:
optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-4
)

In [ ]:
model, train_losses, val_losses, val_accuracies = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=8,
    lr=1e-4,
    save_path="mobilenetv2_finetuned_best.pth"
)

In [ ]:
model.load_state_dict(torch.load("mobilenetv2_finetuned_best.pth"))

test_acc, test_f1 = evaluate_model(model, test_loader)

print("MobileNetV2 Fine-Tuned Results")
print(f"Test Accuracy : {test_acc*100:.2f}%")
print(f"Test Macro F1 : {test_f1:.4f}")

In [ ]:
transform = transforms.Compose([
    transforms.Resize((260, 260)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

## EfficientNet-B2 Model with Change in Dimensions for the image

In [ ]:
dataset_path = "/content/drive/MyDrive/caltech-101/101_ObjectCategories" # Corrected path to mounted Google Drive folder

dataset = datasets.ImageFolder(root=dataset_path, transform=transform)

num_classes = 102

In [ ]:
from sklearn.model_selection import train_test_split

SEED = 42

# Get labels and indices
targets = np.array(dataset.targets)
indices = np.arange(len(targets))

# ---- First split: 70% Train, 30% Temp ----
train_idx, temp_idx = train_test_split(
    indices,
    test_size=0.3,
    stratify=targets,
    random_state=SEED
)

# ---- Second split: 10% Val, 20% Test ----
val_idx, test_idx = train_test_split(
    temp_idx,
    test_size=0.66,
    stratify=targets[temp_idx],
    random_state=SEED
)

In [ ]:
from torch.utils.data import Subset

train_dataset = Subset(dataset, train_idx)
val_dataset   = Subset(dataset, val_idx)
test_dataset  = Subset(dataset, test_idx)

print("Train size:", len(train_dataset))
print("Validation size:", len(val_dataset))
print("Test size:", len(test_dataset))

In [ ]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=32, num_workers = 2, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=32, num_workers = 2, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=32, num_workers = 2, shuffle=False)

In [ ]:
from collections import Counter

train_labels = targets[train_idx]
val_labels   = targets[val_idx]
test_labels  = targets[test_idx]

print("Train distribution:", Counter(train_labels))
print("Val distribution:", Counter(val_labels))
print("Test distribution:", Counter(test_labels))

In [ ]:
import torch
import torch.nn as nn
from torchvision import models

model = models.efficientnet_b2(pretrained=True)

num_classes = len(dataset.classes)

model.classifier[1] = nn.Linear(
    model.classifier[1].in_features,
    num_classes
)

for param in model.parameters():
    param.requires_grad = False

for param in model.classifier.parameters():
    param.requires_grad = True

model = model.to(device)

In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [ ]:
model, train_losses, val_losses, val_accuracies = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=8,
    lr=1e-3,
    save_path="efficientnet_b2_frozen_best.pth"
)

In [ ]:
model.load_state_dict(torch.load("efficientnet_b2_frozen_best.pth"))

test_acc, test_f1 = evaluate_model(model, test_loader)

print("\n===== Test Results (Efficientnet-B2 Frozen) =====")
print(f"Test Accuracy : {test_acc*100:.2f}%")
print(f"Test Macro F1 : {test_f1:.4f}")
print("===========================================")

In [ ]:
model = models.efficientnet_b2(pretrained=True)

num_classes = len(dataset.classes)

model.classifier[1] = nn.Linear(
    model.classifier[1].in_features,
    num_classes
)

for param in model.parameters():
    param.requires_grad = False

for param in model.classifier.parameters():
    param.requires_grad = True

for param in model.features[-1].parameters():
    param.requires_grad = True

for param in model.features[-2].parameters():
    param.requires_grad = True

model = model.to(device)

In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

In [ ]:
model, train_losses, val_losses, val_accuracies = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=8,
    lr=1e-4,
    save_path="efficientnet_b2_finetuned_best.pth"
)

In [ ]:
model.load_state_dict(torch.load("efficientnet_b2_finetuned_best.pth"))

test_acc, test_f1 = evaluate_model(model, test_loader)

print("\n===== Test Results (Efficientnet-B2 Frozen) =====")
print(f"Test Accuracy : {test_acc*100:.2f}%")
print(f"Test Macro F1 : {test_f1:.4f}")
print("===========================================")